# Definitions

## Manual Definition

In [ ]:
from types import SimpleNamespace
overwrite = SimpleNamespace()

# overwrite.model_space = "VAE"
overwrite.model_space = "PX"

overwrite.scale_factor = 1 # PRoPE / LVSM pre-training
# overwrite.scale_factor = 2 # LVSM fine-tuning

overwrite.patch_size = 8 # LVSM / PRoPE
overwrite.patch_size = 1 # option b for VAE
overwrite.patch_size = 32 # option b for PX

# overwrite.supervise_views = 4 # → will result in 4 in, 1 out
overwrite.supervise_views = 0 # → will result in 1 in, 1 out

# overwrite.dim_feedforward = 3072 # LVSM / PRoPE - large
overwrite.dim_feedforward = 1024 # PRoPE - small

overwrite.num_layers = 6 # PRoPE - small
# overwrite.num_layers = 12 # PRoPE - large

# LVSM:
# For scene-level experiments We use 2 input views and 6 target views for each training example.

## Generating all Combinations

In [30]:
from itertools import product
from types import SimpleNamespace

def generate_overwrite_configs():
    model_spaces = ["VAE", "PX"]
    scale_factors = [1, 2]
    supervise_views = [0, 4]
    dim_feedforwards = [1024, 3072]
    num_layers = [6, 12]

    all_configs = []

    for model_space in model_spaces:

        # patch_size depends on model_space
        if model_space == "VAE":
            patch_sizes = [1, 8]
        else:  # PX
            patch_sizes = [8, 32]

        # make cartesian product with allowed settings
        for (scale_factor,
             patch_size,
             sv,
             dff,
             nl) in product(scale_factors,
                            patch_sizes,
                            supervise_views,
                            dim_feedforwards,
                            num_layers):

            ow = SimpleNamespace()
            ow.model_space = model_space
            ow.scale_factor = scale_factor
            ow.patch_size = patch_size
            ow.supervise_views = sv
            ow.dim_feedforward = dff
            ow.num_layers = nl

            all_configs.append(ow)

    return all_configs

# Example usage:
configs = generate_overwrite_configs()
print("Total configs:", len(configs))
print(vars(configs[0]))   # show first


Total configs: 64
{'model_space': 'VAE', 'scale_factor': 1, 'patch_size': 1, 'supervise_views': 0, 'dim_feedforward': 1024, 'num_layers': 6}


# Functions

In [22]:
import sys
import os
project_root = os.path.abspath("..")
sys.path.append(project_root)

import torch
import yaml
import re
import glob
from pathlib import Path

from src.models.lvsm_decoder_only import LVSMDecoderOnlyModel
from src.configs.lvsm_decoder_only_config import (
    LVSMDecoderOnlyModelConfig,
    RayEncodingType,
    PosEncType,
)
from src.prope.utils.transformer import (
    TransformerEncoderConfig,
    TransformerEncoderLayerConfig,
)


from src.data.dataset import TrainDataset
from src.data.latent_dataset import TrainLatentDataset
from main import LVSMLauncher, LVSMLauncherConfig

device = "cuda" if torch.cuda.is_available() else "cpu"


def get_config(overwrite):           
    # "assets/px_config.yaml"
    # dir = "../assets/"
    dir = "../assets/vae_config.yaml" if overwrite.model_space == "VAE" else "../assets/px_config.yaml"
    raw = Path(dir).read_text()
    # raw = Path(f"{dir}/vae_config.yaml").read_text()
    # run_dir = "../results/nvs/release-1gpus-b8-s1-80k-CAMRAY-PROPE/" #VAE
    # raw = Path(f"{run_dir}/config.yaml").read_text()
    clean = re.sub(r"!!python/[^ \n]+", "", raw)   # strip python tags

    config_dict = yaml.safe_load(clean)    
    cfg = LVSMLauncherConfig()

    for k, v in config_dict.items():
        if k != "model_config":   # we'll rebuild this separately
            setattr(cfg, k, v)
    
    m = config_dict["model_config"]

    # Enums were dumped as single-item lists
    ray_encoding = RayEncodingType(m["ray_encoding"][0])
    pos_enc      = PosEncType(m["pos_enc"][0])
    
    layer_cfg = m["encoder"]["layer"]

    encoder_layer = TransformerEncoderLayerConfig(
        d_model=layer_cfg["d_model"],
        nhead=layer_cfg["nhead"],
        dim_feedforward=layer_cfg["dim_feedforward"],
        dropout=layer_cfg["dropout"],
        activation=torch.nn.functional.relu,  # original activation
        batch_first=layer_cfg["batch_first"],
        bias=layer_cfg["bias"],
        layer_norm_eps=layer_cfg["layer_norm_eps"],
        modulation_activation=layer_cfg["modulation_activation"],
        norm_first=layer_cfg["norm_first"],
        norm_type=layer_cfg["norm_type"],
        elementwise_affine=layer_cfg["elementwise_affine"],
        qk_norm=layer_cfg["qk_norm"],
    )

    encoder = TransformerEncoderConfig(
        layer=encoder_layer,
        num_layers=m["encoder"]["num_layers"],
        input_norm=m["encoder"]["input_norm"],
        output_norm=m["encoder"]["output_norm"],
        checkpointing=m["encoder"]["checkpointing"],
    )

    # Full model config
    model_cfg = LVSMDecoderOnlyModelConfig(
        ref_views=m["ref_views"],
        tar_views=m["tar_views"],
        encoder=encoder,
        img_shape=tuple([m["img_shape"][0]*overwrite.scale_factor, m["img_shape"][1]*overwrite.scale_factor,m["img_shape"][2]]),
        cam_shape=tuple([m["cam_shape"][0]*overwrite.scale_factor, m["cam_shape"][1]*overwrite.scale_factor,m["cam_shape"][2]]),        
        patch_size=overwrite.patch_size,        
        ray_encoding=ray_encoding,
        pos_enc=pos_enc,
    )
    
    cfg.model_config = model_cfg
    return cfg

def load_checkpoint(model, dir):
    def get_latest_checkpoint(ckpt_dir):
        ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "*.pt")))
        return ckpts[-1] if ckpts else None

    ckpt_path = get_latest_checkpoint(f"{dir}/ckpts")
    print("Loading checkpoint:", ckpt_path)

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    state_dict = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model"].items()}
    missing, unexpected = model.load_state_dict(state_dict, strict=False)

    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)
    return model



def load_single_batch(cfg: LVSMLauncherConfig, overwrite):
    """
    Loads ONE batch from the appropriate dataset based on cfg.model_space.
    Returns ref_imgs, tar_imgs, ref_cams, tar_cams, ref_paths, tar_paths.
    """

    launcher = LVSMLauncher(cfg)
    
    if cfg.model_space == "PX":
        # scenes = sorted(glob.glob("../data/data_processed/realestate10k/train/*"))
        scenes = sorted(glob.glob("../data/data_processed/realestate10k/overfit/*"))
        dataset = TrainDataset(
            scenes,
            patch_size=cfg.dataset_patch_size,
            zoom_factor=cfg.train_zoom_factor,
            random_zoom=cfg.random_zoom,
            # supervise_views=cfg.dataset_supervise_views,
            supervise_views=overwrite.supervise_views,
            upscale_factor=overwrite.scale_factor
        )
    else:
        # scenes = sorted(glob.glob("../data/data_processed/realestate10k_latent/train/*"))
        scenes = sorted(glob.glob("../data/data_processed/realestate10k_latent/overfit/*"))
        dataset = TrainLatentDataset(
            scenes,
            # supervise_views=cfg.dataset_supervise_views,     
            supervise_views=overwrite.supervise_views,       
            upscale_factor=overwrite.scale_factor
        )    

    data = dataset[0] # just one item
    data["image"] = data["image"].unsqueeze(0)
    data["K"] = data["K"].unsqueeze(0)
    data["camtoworld"] = data["camtoworld"].unsqueeze(0)
    data["image_path"] = [data["image_path"]]  # wrap in list    

    input_views = data["K"].shape[1] - cfg.dataset_supervise_views
    
    processed = launcher.preprocess(data, input_views=input_views)

    ref_imgs = processed["ref_imgs"]
    tar_imgs = processed["tar_imgs"]
    ref_cams = processed["ref_cams"]
    tar_cams = processed["tar_cams"]
    ref_paths = processed["ref_paths"]
    tar_paths = processed["tar_paths"]

    return ref_imgs, tar_imgs, ref_cams, tar_cams, ref_paths, tar_paths



import time
import torch
import torch.nn.functional as F

def tensor_bytes(x: torch.Tensor) -> int:
    return x.numel() * x.element_size()

def encode_tensors(img: torch.Tensor, vae, device) -> torch.Tensor:
    """
    Accepts:
        [H, W, 3]
        [V, H, W, 3]
        [B, V, H, W, 3]

    Returns:
        Same leading dims, but encoded latents:
        [.., H_down, W_down, C_latent]
    """

    img = img.to(device).float()

    # Original leading dims: (), (V), or (B, V)
    orig_shape = img.shape[:-3]
    H, W, C = img.shape[-3:]
    assert C == 3, f"Expected RGB input but got {C} channels"

    # Flatten leading dims
    img = img.reshape(-1, H, W, C)        # [N, H, W, 3]

    # Convert to NCHW
    img = img.permute(0, 3, 1, 2)         # [N, 3, H, W]

    # Scale to [-1, 1] just like training
    img = img * 2 - 1

    with torch.no_grad():
        posterior = vae.encode(img)
        latents = posterior.latent_dist.mean   # [N, C_lat, H_down, W_down]

    # Back to NHWC
    latents = latents.permute(0, 2, 3, 1)   # [N, H_down, W_down, C_lat]


    # Restore original leading dims
    latents = latents.reshape(*orig_shape,
                              latents.shape[1],
                              latents.shape[2],
                              latents.shape[3])

    return latents


def scale_spatial(img: torch.Tensor, factor: int) -> torch.Tensor:
    """
    Inputs:
        img: [B, V, H, W, C]
    Returns:
        [B, V, H*factor, W*factor, C]

    Uses bilinear interpolation. Assumes img is on any device.
    """

    B, V, H, W, C = img.shape
    img_nchw = img.permute(0, 1, 4, 2, 3).reshape(B*V, C, H, W)

    img_up = F.interpolate(
        img_nchw,
        size=(H * factor, W * factor),
        mode="bilinear",
        align_corners=False
    )
    img_up = img_up.reshape(B, V, C, H*factor, W*factor).permute(0, 1, 3, 4, 2)

    return img_up


def run_inference_with_metrics(model, launcher, ref_imgs, ref_cams, tar_cams, cfg, device="cuda", scale_factor=1):    

    model_params = sum(p.numel() for p in model.parameters())
    
    input_tensors = [ref_imgs]
    input_bytes = sum(tensor_bytes(t) for t in input_tensors if isinstance(t, torch.Tensor))    
    
    forward_memory_mb = 0.0
    encode_memory_mb = 0.0
    decode_memory_mb = 0.0
    peak_memory_mb = 0.0
    encode_peak = 0.0
    encode_time_ms = 0.0

    t_enc1 = time.time()
    

    if cfg.model_space == "VAE":
        decoded = launcher.decode_tensors(ref_imgs)       
        print(f"ref_imgs.shape {ref_imgs.shape}")             
        if scale_factor > 1:
            decoded = scale_spatial(decoded, scale_factor)
        t_enc1 = time.time()
        encode_tensors(decoded, launcher.vae32, device)
        encode_time_ms = (time.time() - t_enc1)* 1000
        if torch.cuda.is_available():
            encode_memory_mb = torch.cuda.max_memory_allocated(device) / (1024**2)
        encode_peak = torch.cuda.max_memory_allocated(device)

    # ------------------------------------------
    # Forward pass timing + memory
    # ------------------------------------------
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    torch.cuda.synchronize()
    t_fwd0 = time.time()

    print(f"ref_imgs.shape {ref_imgs.shape}")
    print(f"ref_cams.shape {len(ref_cams)}")
    print(f"tar_cams.shape {len(tar_cams)}")
    with torch.inference_mode():
        out = model(ref_imgs.to(device), ref_cams, tar_cams)

    torch.cuda.synchronize()
    t_fwd1 = time.time()

    forward_time_ms = (t_fwd1 - t_fwd0) * 1000

    # Memory AFTER forward pass
    if torch.cuda.is_available():
        forward_memory_mb = torch.cuda.max_memory_allocated(device) / (1024**2)

    # Keep the forward peak BEFORE decoding wipes it out
    forward_peak = torch.cuda.max_memory_allocated(device)
    
    if cfg.model_space == "VAE":

        # Reset for decode-only measurement
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize()
        t_dec0 = time.time()

        decoded = launcher.decode_tensors(out[0])

        torch.cuda.synchronize()
        t_dec1 = time.time()
        decode_time_ms = (t_dec1 - t_dec0) * 1000

        decode_memory_mb = torch.cuda.max_memory_allocated(device) / (1024**2)

    else:
        decoded = torch.sigmoid(out)
        decode_time_ms = 0.0
        decode_memory_mb = 0.0
    
    peak_memory_mb = max(encode_peak, forward_peak, torch.cuda.max_memory_allocated(device)) / (1024**2)    
    total_time_ms = forward_time_ms + decode_time_ms + encode_time_ms


    return {
        "model_space": cfg.model_space,
        "patch_size" : cfg.model_config.patch_size,
        "layers": cfg.model_config.encoder.num_layers,
        "dim_feedforward": cfg.model_config.encoder.layer.dim_feedforward,
        "scale_factor" : cfg.upscale,        
        "ref_views" : ref_imgs.shape[1],
        "model_params": model_params,
        "input_bytes": input_bytes,
        "input_MB": input_bytes / (1024**2),
        "encode_time_ms": encode_time_ms,
        "forward_time_ms": forward_time_ms,
        "decode_time_ms": decode_time_ms,
        "total_time_ms": total_time_ms,
        "encode_memory_mb": encode_memory_mb,
        "forward_memory_mb": forward_memory_mb,
        "decode_memory_mb": decode_memory_mb,
        "peak_memory_mb": peak_memory_mb,
        # "output_shape": tuple(out.shape),
        # "decoded_shape": tuple(decoded.shape),
    }


In [ ]:

def evaluate(config):
    cfg = get_config(config)
    model = LVSMDecoderOnlyModel(cfg.model_config).to(device)
    model.eval()

    ref_imgs, tar_imgs, ref_cams, tar_cams, ref_paths, tar_paths = load_single_batch(cfg, config)

    launcher = LVSMLauncher(cfg)
    return run_inference_with_metrics(
        model=model,
        launcher=launcher,
        ref_imgs=ref_imgs,
        ref_cams=ref_cams,
        tar_cams=tar_cams,
        cfg=cfg,
        scale_factor=overwrite.scale_factor
    )

In [ ]:
results = []

import json

n_iterations = 10

for config in configs[0]:
    for n in range(n_iterations):
        metrics = evaluate(config)
        results.append(metrics)        

# metrics = evaluate(configs[0])
# results.append(metrics)   
# print(json.dumps(metrics, indent=4))

with open("../results/performance_comparison/results.json", "w") as f:
    json.dump(results,f,indent=4)


Wrote config to results/nvs/VAE/release-1gpus-b8-s1-80k-CAMRAY-PROPE/config.yaml
training on latent dataset
Wrote config to results/nvs/VAE/release-1gpus-b8-s1-80k-CAMRAY-PROPE/config.yaml
ref_imgs.shape torch.Size([1, 1, 32, 32, 16])


/home/felix/LVSM-VAE/src/data/latent_dataset.py:243: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  image = torch.load(abs_image_path)


ref_imgs.shape torch.Size([1, 1, 32, 32, 16])
ref_cams.shape 4
tar_cams.shape 4


# Plots

In [24]:
# time-memory scatter plot

import matplotlib.pyplot as plt

# total_time = [d["total_time_ms"] for d in collection]
# peak_mem = [d["peak_memory_mb"] for d in collection]

# plt.figure(figsize=(6,4))
# plt.scatter(total_time, peak_mem)
# plt.xlabel("total_time_ms")
# plt.ylabel("peak_memory_mb")
# plt.title("Scatter Plot of Total Time vs Peak Memory")
# plt.tight_layout()
# plt.show()



In [25]:

import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
